# Text to Music Generation (MusicGen)

Generates a short music clip from a text description. This model can be large and is much more practical with a GPU. Check model license and usage terms.

In [ ]:
%pip install -q transformers scipy torch

In [ ]:
import torch
import scipy.io.wavfile as wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration

model_id = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Loaded on", device)

In [ ]:
prompt = input("Enter a music description: ").strip()
if not prompt:
    raise ValueError("Please enter a description.")
inputs = processor(text=[prompt], padding=True, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}
with torch.inference_mode():
    audio_values = model.generate(**inputs, max_new_tokens=256)
audio = audio_values[0, 0].detach().cpu().float().numpy()
sample_rate = model.config.audio_encoder.sampling_rate
wavfile.write("generated_music.wav", sample_rate, audio)
print("Music saved as generated_music.wav")
display(__import__("IPython").display.Audio(audio, rate=sample_rate))